# Time Series, Forecasting and Agentic Pipelines

Your Name

**Topics Covered:** Time Series Analysis · Autoregression · ARIMA · Prophet · Sklearn Lag Models · Smolagents · GROQ · HuggingFace

**Libraries:** `yfinance` · `statsmodels` · `sklearn` · `prophet` · `smolagents` · `groq` · `huggingface_hub`

> **Learning Goal:** By the end of this notebook you will have built a full forecasting pipeline — from live market data to a working AI agent that analyzes and narrates your results — skills directly applicable in industry interviews and production systems.

---

## Getting Started

- Replace **Your Name** above with your full name
- Automatic 0 if you include your student ID or any other identifying number
- Rename the file to `Your_Name.ipynb` before submitting
- When finished, share your Colab link with **Anyone with the link can view** privileges
- Paste the shared link into the Canvas submission
- Get your Groq API key and HuggingFace token and store them in Colab Secrets (not hardcoded in the notebook)

> **Setup task:** Sign up at [console.groq.com](https://console.groq.com) and [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) for free API keys.

---

## How to Use This Notebook

This notebook is designed to be accessible to all users, including those navigating with a **screen reader**.

### Screen Reader Navigation Guide

Every code section in this notebook gives you a choice. Before each code cell you will find a navigation landmark with two options:

- **Skip this code cell** — Jumps past the code directly to the output explanation.
- **Go back and read the code cell** — Returns you to the top of the code section.

**Tips for screen reader users:**
- Press **H** to jump between major section headings
- Press **K** or **Tab** to move between links, including skip and return navigation links
- Press **D** to jump between landmark regions

---

## Learning Objectives

By the end of this assignment you will be able to:

1. Acquire and explore a live financial time series using `yfinance`
2. Test for stationarity using the Augmented Dickey-Fuller test
3. Interpret ACF and PACF plots to select ARIMA order parameters
4. Fit and evaluate an ARIMA model using walk-forward validation
5. Re-frame a time series forecasting problem as supervised regression using lag features
6. Compare multiple forecasting models using MAE, RMSE, and MAPE
7. Fit a Prophet model with uncertainty intervals and interpret its components
8. Build an agentic forecasting pipeline using Smolagents, GROQ, and HuggingFace
9. Explain the trade-offs between LLM providers in a production system

---

## Notebook Roadmap

| Section | Topic | Style |
| :--- | :--- | :--- |
| 1 | Environment Setup | Guided |
| 2 | Data Acquisition and EDA | Fill-in-the-blank |
| 3 | Stationarity, Decomposition, and Autocorrelation | Fill-in-the-blank |
| 4 | ARIMA with statsmodels | Fill-in-the-blank then complete |
| 5 | Lag-Feature Regression with sklearn | Guided then complete |
| 6 | Forecasting with Prophet | Complete with discussion |
| 7 | Agentic Pipeline with Smolagents, GROQ, and HuggingFace | Complete with discussion |

---

---
# Section 1 — Environment Setup

Before doing any data science, we need a clean, reproducible environment. This mirrors industry practice — every production ML project starts with dependency management. Run both cells in this section first.

<a name="read-code-1"></a>

**Cell 1 — Install All Required Packages**

This cell uses Python's `subprocess` module to install all required libraries via pip: `yfinance` for market data, `statsmodels` for ARIMA, `prophet` for Meta's forecasting library, `smolagents` for the agentic pipeline, `groq` and `huggingface_hub` for LLM backends, and the standard data science stack. Run this cell first — if prompted to restart the kernel, do so before continuing.

<nav aria-label="Code cell 1 navigation">
<a href="#skip-code-1" aria-label="Skip code cell 1 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import subprocess, sys

packages = [
    "yfinance", "statsmodels", "scikit-learn", "prophet",
    "smolagents", "groq", "huggingface_hub",
    "plotly", "pandas", "numpy", "matplotlib", "seaborn",
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print("All packages installed successfully.")

<a name="skip-code-1"></a>

**Expected output:** `All packages installed successfully.`

If a package fails, note the error message and try installing it manually with `!pip install <package_name>` in a new cell.

<nav aria-label="Return navigation for code cell 1">
<a href="#read-code-1" aria-label="Go back and read code cell 1">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-2"></a>

**Cell 2 — Core Imports and Display Settings**

This cell imports all libraries and configures display settings. Key imports include `yfinance` for live market data, `statsmodels` modules for ADF testing, decomposition, and ARIMA, `prophet` for the Meta forecasting library, and `smolagents` plus `groq` for the agentic pipeline in Section 7.

<nav aria-label="Code cell 2 navigation">
<a href="#skip-code-2" aria-label="Skip code cell 2 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Time series
import yfinance as yf
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Sklearn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Prophet
from prophet import Prophet
from prophet.plot import plot_plotly

# API clients
from groq import Groq
from huggingface_hub import InferenceClient

# Smolagents
from smolagents import tool, CodeAgent, InferenceClientModel

pd.set_option("display.float_format", "{:.4f}".format)
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

print("All imports successful.")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")

<a name="skip-code-2"></a>

**Expected output:** `All imports successful.` followed by version numbers for pandas and numpy.

<nav aria-label="Return navigation for code cell 2">
<a href="#read-code-2" aria-label="Go back and read code cell 2">&#8617; Go back and read the code cell</a>
</nav>

### API Key Setup

We use two free API providers:

- **GROQ** — blazing-fast LLM inference (free tier, no credit card). Sign up at [console.groq.com](https://console.groq.com)
- **HuggingFace** — free inference API for open-source models. Get your token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

> **Never hardcode API keys in a notebook you share.** Use Colab Secrets (the key icon in the left sidebar): add `GROQ_API_KEY` and `HF_TOKEN` as secrets, then enable notebook access for each.

<a name="read-code-3"></a>

**Cell 3 — Load API Keys from Colab Secrets**

This cell loads both API keys from Colab Secrets into environment variables. The keys are never printed in full — only the first 8 characters are shown for verification. If you have not set up your secrets yet, go to the key icon in the left Colab sidebar, add `GROQ_API_KEY` and `HF_TOKEN`, and enable notebook access for each.

<nav aria-label="Code cell 3 navigation">
<a href="#skip-code-3" aria-label="Skip code cell 3 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
import os
from google.colab import userdata

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

GROQ_API_KEY = os.environ["GROQ_API_KEY"]
HF_TOKEN     = os.environ["HF_TOKEN"]

print("API keys loaded.")
print("   GROQ key starts with:", GROQ_API_KEY[:8], "...")
print("   HF   key starts with:", HF_TOKEN[:8], "...")

<a name="skip-code-3"></a>

**Expected output:** Both key prefixes printed. If you see a `SecretNotFoundError`, confirm you added the secrets in the Colab sidebar and enabled notebook access for each one.

<nav aria-label="Return navigation for code cell 3">
<a href="#read-code-3" aria-label="Go back and read code cell 3">&#8617; Go back and read the code cell</a>
</nav>

---
# Section 2 — Data Acquisition and Exploratory Analysis

We will work with **NVIDIA Corporation (NVDA)** daily closing prices — a real financial time series that captures one of the most dramatic growth stories in recent market history. NVIDIA's stock reflects the AI hardware boom: a decade of steady growth, the COVID volatility shock of 2020, a crypto-driven spike, a deep correction in 2022, and then an extraordinary AI-driven surge from 2023 onward. This makes it structurally rich, highly non-stationary, and perfect for learning time series diagnostics.

This is exactly the kind of data you will encounter in quantitative finance, data engineering, and ML engineering roles.

---

<a name="read-code-4"></a>

**Cell 4 — Download NVDA Closing Price History**

This cell uses `yfinance` to download NVIDIA daily closing prices from January 2018 through December 2024. It saves a local fallback CSV in case Yahoo Finance is temporarily unavailable. The result is a single-column DataFrame with dates as the index and `price` as the column name.

<nav aria-label="Code cell 4 navigation">
<a href="#skip-code-4" aria-label="Skip code cell 4 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
TICKER   = "NVDA"
START    = "2018-01-01"
END      = "2024-12-31"
FALLBACK = "nvda_fallback.csv"

try:
    raw = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)
    if raw.empty:
        raise ValueError('Empty response from yfinance')
    raw.to_csv(FALLBACK)
    print(f"Downloaded {len(raw)} rows from Yahoo Finance.")
except Exception as e:
    print(f"yfinance failed ({e}). Loading fallback CSV.")
    raw = pd.read_csv(FALLBACK, index_col=0, parse_dates=True)

df = raw[["Close"]].copy()
df.columns = ["price"]
df.index = pd.to_datetime(df.index)
df.index.name = "date"

print(f"Dataset shape : {df.shape}")
print(f"Date range    : {df.index.min().date()} -> {df.index.max().date()}")
df.head(10)

<a name="skip-code-4"></a>

**Expected output:** Approximately 1,760 rows of NVDA price data covering 2018 through 2024, followed by a 10-row preview. The price should start around $5–6 in 2018 (pre-split adjusted) and end above $130 in late 2024.

<nav aria-label="Return navigation for code cell 4">
<a href="#read-code-4" aria-label="Go back and read code cell 4">&#8617; Go back and read the code cell</a>
</nav>

### 2.2 Fill-in-the-Blank: Basic Exploration

> **Exercise** — Complete the cells below. Replace every `___` with the correct value or method call. Use pandas methods such as `.describe()`, `.isnull()`, and `.resample()`.

<a name="read-code-5"></a>

**Cell 5 — Summary Statistics (Fill in the Blank)**

Call the pandas method on `df['price']` that returns count, mean, std, min, and max. Replace the `___` with the correct method name.

<nav aria-label="Code cell 5 navigation">
<a href="#skip-code-5" aria-label="Skip code cell 5 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# FILL IN: call the pandas method that returns count, mean, std, min, max
summary = df["price"].___()
print(summary)

<a name="skip-code-5"></a>

**Answer:** Replace `___` with `describe`. The output should show NVDA's dramatic price range — the max will be substantially higher than the mean, reflecting the AI-driven surge concentrated in the last 18 months of the dataset.

<nav aria-label="Return navigation for code cell 5">
<a href="#read-code-5" aria-label="Go back and read code cell 5">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-6"></a>

**Cell 6 — Check and Fill Missing Values (Fill in the Blank)**

Count the nulls in the price column, then forward-fill any gaps using `.ffill()`. Financial time series sometimes have missing trading day entries — forward-filling carries the last known price forward.

<nav aria-label="Code cell 6 navigation">
<a href="#skip-code-6" aria-label="Skip code cell 6 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# FILL IN: count how many nulls are in the price column
missing = df["price"].___().___()
df["price"] = df["price"].___()
print(f"Missing values before fill: {missing}")
print("After fill:", df["price"].isnull().sum(), "missing values remain.")

<a name="skip-code-6"></a>

**Answer:** The three blanks are `.isnull()`, `.sum()`, and `.ffill()`. Missing value counts are typically very low (0–5) for major tickers like NVDA.

<nav aria-label="Return navigation for code cell 6">
<a href="#read-code-6" aria-label="Go back and read code cell 6">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-7"></a>

**Cell 7 — Monthly Average Price (Fill in the Blank)**

Resample the daily price series to monthly frequency and compute the mean price per month. Replace the `___` inside `resample()` with the correct pandas frequency alias for month-end.

<nav aria-label="Code cell 7 navigation">
<a href="#skip-code-7" aria-label="Skip code cell 7 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# FILL IN: resample to monthly frequency and compute the mean
# Hint: the monthly end frequency alias is a single capital letter
monthly = df["price"].resample(___).___()
print(monthly.tail(12))

<a name="skip-code-7"></a>

**Answer:** Replace `___` with `'ME'` (month-end) and add `.mean()`. The last 12 rows should show NVDA's monthly average climbing steeply from late 2023 onward — the visual signature of the AI hardware demand surge.

<nav aria-label="Return navigation for code cell 7">
<a href="#read-code-7" aria-label="Go back and read code cell 7">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-8"></a>

**Cell 8 — Full Price History with Moving Averages and Daily Returns**

This cell computes 50-day and 200-day moving averages and plots two panels: the price history with moving average overlays (top) and daily percentage returns (bottom). The moving average crossover pattern (where MA50 crosses above MA200) is a classic technical signal.

<nav aria-label="Code cell 8 navigation">
<a href="#skip-code-8" aria-label="Skip code cell 8 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
df["MA50"]  = df["price"].rolling(window=50).mean()
df["MA200"] = df["price"].rolling(window=200).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df.index, df["price"], label="Daily Close", alpha=0.7, linewidth=1)
axes[0].plot(df.index, df["MA50"],  label="50-day MA",   linewidth=1.5, linestyle="--")
axes[0].plot(df.index, df["MA200"], label="200-day MA",  linewidth=1.5, linestyle="-.")
axes[0].set_title(f"{TICKER} Daily Closing Price with Moving Averages", fontsize=14)
axes[0].set_ylabel("Price (USD)")
axes[0].legend()

df["returns"] = df["price"].pct_change()
axes[1].bar(df.index, df["returns"], alpha=0.5, color="steelblue", width=1)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Daily Returns (%)", fontsize=14)
axes[1].set_ylabel("Return")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.savefig("nvda_overview.png", dpi=150)
plt.show()
print("Chart saved.")

<a name="skip-code-8"></a>

**Expected output:** A two-panel chart. The top panel should show a gradual upward trend through 2021, a deep correction in 2022, and then a near-vertical climb from mid-2023 onward as NVDA became the central infrastructure stock of the AI boom. The bottom panel (daily returns) will show elevated volatility during the 2022 correction and 2023 rally.

**Accessibility note:** Add a Markdown cell below describing both panels — note the 2022 trough, the 2023 breakout, and whether you can spot a Golden Cross (MA50 crossing above MA200) anywhere in the chart.

<nav aria-label="Return navigation for code cell 8">
<a href="#read-code-8" aria-label="Go back and read code cell 8">&#8617; Go back and read the code cell</a>
</nav>

### Discussion Question 2A

> Look at the chart above and answer the following questions.

> 1. Can you identify any obvious trend? Is this series stationary? What does non-stationarity mean for the classical modeling approaches you will use in Sections 3 and 4?

> 2. NVDA's daily returns distribution is likely to have fat tails — more extreme days than a normal distribution would predict. Why does this matter for financial risk models that assume normality?

> 3. NVDA's 2023 surge was closely tied to the announcement of AI GPU demand from hyperscalers like Microsoft and Google. Identify approximately when this structural break occurred in the price series. How would a structural break like this affect an ARIMA model trained on data from before the break?

**Write your answers below (double-click this cell to edit):**

1.

2.

3.

---
# Section 3 — Stationarity, Decomposition, and Autocorrelation

This section covers the **diagnostic layer** of time series analysis — the work that must happen before any model is fit. These concepts appear constantly in quantitative data science interviews.

| Concept | Definition | Why It Matters |
| :--- | :--- | :--- |
| **Stationarity** | Statistical properties (mean, variance) constant over time | Most classical models assume it |
| **Decomposition** | Splitting into Trend + Seasonality + Residual | Reveals structure, aids model selection |
| **ACF** | Correlation of a series with its own lagged values | Tells you how far back to look |
| **PACF** | ACF after removing shorter-lag effects | Helps select AR order |
| **ADF Test** | Augmented Dickey-Fuller — formal unit root test | p < 0.05 means stationary |

---

<a name="read-code-9"></a>

**Cell 9 — ADF Stationarity Test on Raw NVDA Price (Fill in the Blank)**

This cell defines a reusable `adf_report()` function that runs the Augmented Dickey-Fuller test and prints a clean summary including the test statistic, p-value, number of lags, critical values, and a plain-language conclusion. Fill in the blank: pass the correct column name to test the raw price series.

<nav aria-label="Code cell 9 navigation">
<a href="#skip-code-9" aria-label="Skip code cell 9 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
def adf_report(series, label='Series'):
    result = adfuller(series.dropna())
    print('\n' + '='*50)
    print(f'ADF Test: {label}')
    print('='*50)
    print(f'  Test Statistic : {result[0]:.4f}')
    print(f'  p-value        : {result[1]:.6f}')
    print(f'  Lags Used      : {result[2]}')
    print('  Critical Values:')
    for key, val in result[4].items():
        print(f'    {key}: {val:.4f}')
    conclusion = 'STATIONARY (reject H0)' if result[1] < 0.05 else 'NON-STATIONARY (fail to reject H0)'
    print(f'  Conclusion     : {conclusion}')
    return result[1]

# FILL IN: replace ___ with the correct column name
p_raw = adf_report(df[___], label='NVDA Raw Price')

<a name="skip-code-9"></a>

**Answer:** Replace `___` with `'price'`.

**Expected output:** A p-value close to 1.0 (well above 0.05) and a conclusion of NON-STATIONARY. NVDA's strong upward trend means the mean is clearly not constant over time — a direct violation of the stationarity assumption.

<nav aria-label="Return navigation for code cell 9">
<a href="#read-code-9" aria-label="Go back and read code cell 9">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-10"></a>

**Cell 10 — First-Order Differencing (Fill in the Blank)**

First-order differencing computes the day-over-day price change, removing the trend. This is the 'I' (Integrated) component of ARIMA. Fill in the blank to compute the first difference and run the ADF test on the differenced series.

<nav aria-label="Code cell 10 navigation">
<a href="#skip-code-10" aria-label="Skip code cell 10 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
# FILL IN: compute the first difference of the price column
# Hint: .diff(1) computes the period-over-period change
df['price_diff'] = df['price'].___(periods=___)

p_diff = adf_report(df['price_diff'].dropna(), label='NVDA First Difference')

print(f'\nRaw series p-value:        {p_raw:.4f}  (non-stationary)')
print(f'Differenced series p-value: {p_diff:.6f}  (stationary)')

<a name="skip-code-10"></a>

**Answer:** `.diff(periods=1)`. The differenced series represents daily price *changes* rather than raw prices. **Expected output:** p-value close to 0 (well below 0.05) — stationary. One round of differencing is sufficient, so `d = 1` for the ARIMA model in Section 4.

<nav aria-label="Return navigation for code cell 10">
<a href="#read-code-10" aria-label="Go back and read code cell 10">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-11"></a>

**Cell 11 — Seasonal Decomposition**

This cell decomposes the price series into four components: Observed (raw), Trend (long-run direction), Seasonal (recurring annual pattern), and Residual (unexplained noise). We use an additive model with `period=252` (approximately 252 trading days per year). For NVDA, pay close attention to the Trend and Residual — the Trend will show the AI-driven acceleration, and the Residual will spike around major news events.

<nav aria-label="Code cell 11 navigation">
<a href="#skip-code-11" aria-label="Skip code cell 11 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
decomp = seasonal_decompose(df['price'].dropna(), model='additive', period=252)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
components = {'Observed': decomp.observed, 'Trend': decomp.trend,
              'Seasonal': decomp.seasonal, 'Residual': decomp.resid}
colors = ['steelblue', 'darkorange', 'green', 'red']

for ax, (name, data), color in zip(axes, components.items(), colors):
    ax.plot(data, color=color, linewidth=0.8)
    ax.set_ylabel(name)
    ax.set_title(name, fontsize=11)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.suptitle(f"{TICKER} Seasonal Decomposition (Additive, period=252)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('nvda_decomp.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-11"></a>

**Expected output:** Four-panel decomposition chart. The Trend panel will show a long plateau through 2020–2022 followed by a steep acceleration from 2023. The Seasonal panel will show relatively small recurring annual patterns. The Residual will show elevated noise during the 2022 correction and the 2023 AI rally.

**Accessibility note:** Add a Markdown cell below describing all four panels. Note approximately when the trend acceleration begins in the Trend panel.

<nav aria-label="Return navigation for code cell 11">
<a href="#read-code-11" aria-label="Go back and read code cell 11">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-12"></a>

**Cell 12 — ACF and PACF Plots on the Differenced Series**

ACF (Autocorrelation Function) shows the correlation between the series and its own lagged values. PACF (Partial Autocorrelation Function) shows the same but removes the influence of intermediate lags. Together they help select the p (AR order) and q (MA order) parameters for ARIMA. Rule of thumb: look for the lag where bars first drop inside the confidence band (dashed blue lines).

<nav aria-label="Code cell 12 navigation">
<a href="#skip-code-12" aria-label="Skip code cell 12 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(df['price_diff'].dropna(),  lags=40, ax=axes[0],
         title='ACF — First-Differenced NVDA Price', alpha=0.05)
plot_pacf(df['price_diff'].dropna(), lags=40, ax=axes[1],
         title='PACF — First-Differenced NVDA Price', alpha=0.05, method='ywm')

plt.tight_layout()
plt.savefig('nvda_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-12"></a>

**Expected output:** Two plots. For a near-random-walk like NVDA's differenced series, most ACF and PACF bars will fall inside the confidence band, suggesting p = 1 and q = 1 as reasonable starting orders for ARIMA(1,1,1).

**Accessibility note:** Add a Markdown cell below describing both plots. Note the first lag where bars consistently fall inside the confidence bands.

<nav aria-label="Return navigation for code cell 12">
<a href="#read-code-12" aria-label="Go back and read code cell 12">&#8617; Go back and read the code cell</a>
</nav>

### Discussion Question 3A

> 1. The raw NVDA price failed the ADF test but the differenced series passed. Explain in plain English why differencing makes a series stationary and what information is lost in the process.

> 2. Looking at the ACF and PACF plots of the differenced series, what values of p (AR order) and q (MA order) would you propose for an ARIMA model? Justify your answer based on the plot.

> 3. The decomposition used `period=252`. Why 252 specifically? What would you change if you were working with hourly trading data rather than daily closing prices?

**Write your answers below:**

1.

2.

3.

---
# Section 4 — ARIMA Modeling with statsmodels

ARIMA stands for **AutoRegressive Integrated Moving Average**. It is the backbone of classical time series forecasting and appears in nearly every quantitative data science interview.

| Parameter | Name | Meaning |
| :--- | :--- | :--- |
| **p** | AR order | How many lagged values of the series to use |
| **d** | Differencing | How many times to difference to achieve stationarity |
| **q** | MA order | How many lagged forecast errors to use |

From Section 3 we know `d = 1`. We start with `p = 1, q = 1`.

---

<a name="read-code-13"></a>

**Cell 13 — Chronological Train / Test Split**

Time series data must NEVER be split randomly. Order is everything — future data leaking into training is one of the most common and costly mistakes in production ML. We use a strict chronological cut: everything before 2023-01-01 trains the model, everything after is the test set. A visualization confirms the split visually.

<nav aria-label="Code cell 13 navigation">
<a href="#skip-code-13" aria-label="Skip code cell 13 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
SPLIT_DATE = "2023-01-01"

train = df["price"][:SPLIT_DATE]
test  = df["price"][SPLIT_DATE:]

print(f"Training set : {train.index.min().date()} -> {train.index.max().date()} ({len(train)} days)")
print(f"Test set     : {test.index.min().date()} -> {test.index.max().date()} ({len(test)} days)")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index, train, label="Training", color="steelblue")
ax.plot(test.index,  test,  label="Test",     color="darkorange")
ax.axvline(pd.Timestamp(SPLIT_DATE), color="red", linestyle="--", label="Split Date")
ax.set_title("Train / Test Split — NVDA", fontsize=14)
ax.set_ylabel("Price (USD)")
ax.legend()
plt.tight_layout()
plt.show()

<a name="skip-code-13"></a>

**Expected output:** Training set of approximately 1,260 days and test set of approximately 500 days, plus a chart where the orange (test) line captures NVDA's dramatic AI-driven surge from 2023 onward. This is intentionally challenging for ARIMA — a model trained before the structural break will struggle with the new regime.

<nav aria-label="Return navigation for code cell 13">
<a href="#read-code-13" aria-label="Go back and read code cell 13">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-14"></a>

**Cell 14 — Fit ARIMA(1,1,1)**

This cell fits an ARIMA(1,1,1) model to the training data. The `ValueWarning` suppression is necessary because `yfinance` indices are irregular (market holidays mean there are gaps). The model summary will show coefficient estimates, standard errors, t-statistics, and p-values.

<nav aria-label="Code cell 14 navigation">
<a href="#skip-code-14" aria-label="Skip code cell 14 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ValueWarning

warnings.filterwarnings('ignore', category=ValueWarning)

arima_model  = ARIMA(train, order=(1, 1, 1))
arima_result = arima_model.fit()

print(arima_result.summary())

<a name="skip-code-14"></a>

**Expected output:** The ARIMA model summary table. Key columns to read: the `coef` column (estimated AR and MA coefficients), the `P>|z|` column (are the coefficients statistically significant at p < 0.05?), and the AIC/BIC information criteria at the top (lower = better fit, used for model comparison).

<nav aria-label="Return navigation for code cell 14">
<a href="#read-code-14" aria-label="Go back and read code cell 14">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-15"></a>

**Cell 15 — Residual Diagnostics**

A well-fit ARIMA model leaves residuals that behave like white noise — random, centered at zero, with no remaining autocorrelation. The diagnostic plot shows four panels: residuals over time, histogram, Q-Q plot, and correlogram. The Ljung-Box test formally tests whether any autocorrelation remains in the residuals.

<nav aria-label="Code cell 15 navigation">
<a href="#skip-code-15" aria-label="Skip code cell 15 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
arima_result.plot_diagnostics(figsize=(14, 8))
plt.suptitle('ARIMA(1,1,1) Residual Diagnostics — NVDA', fontsize=13)
plt.tight_layout()
plt.show()

from statsmodels.stats.diagnostic import acorr_ljungbox
lb_result = acorr_ljungbox(arima_result.resid, lags=[10, 20], return_df=True)
print('\nLjung-Box Test (H0: residuals are white noise):')
print(lb_result)
print('\np > 0.05 at all lags = residuals look like white noise (good fit)')

<a name="skip-code-15"></a>

**Expected output:** Four diagnostic panels and a Ljung-Box table. If the Ljung-Box p-values are above 0.05 at all lags, the residuals pass the white noise test. For NVDA, the residuals may show some heteroscedasticity (varying volatility) — the 2022 correction and 2023 rally had very different volatility regimes.

**Accessibility note:** Add a Markdown cell describing what each of the four diagnostic panels shows.

<nav aria-label="Return navigation for code cell 15">
<a href="#read-code-15" aria-label="Go back and read code cell 15">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-16"></a>

**Cell 16 — Walk-Forward Validation (Rolling Forecast)**

Walk-forward validation is the gold standard for evaluating time series models. At each step: fit on all available history up to that point, predict one step ahead, add the actual value to history, and repeat. This mimics a real production system and avoids look-ahead bias. This cell takes several minutes to run — each of the ~500 test steps requires refitting ARIMA.

<nav aria-label="Code cell 16 navigation">
<a href="#skip-code-16" aria-label="Skip code cell 16 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
print('Running walk-forward validation (this will take a few minutes)...')

history     = list(train)
predictions = []
actuals     = list(test)

for i, actual in enumerate(actuals):
    model = ARIMA(history, order=(1, 1, 1))
    fit   = model.fit()
    yhat  = fit.forecast(steps=1)[0]
    predictions.append(yhat)
    history.append(actual)
    if i % 50 == 0:
        print(f'  Step {i+1}/{len(actuals)} predicted: {yhat:.2f}, actual: {actual:.2f}')

predictions = np.array(predictions)
actuals     = np.array(actuals)

mae  = mean_absolute_error(actuals, predictions)
rmse = np.sqrt(mean_squared_error(actuals, predictions))
mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100

print('ARIMA(1,1,1) Walk-Forward Results')
print(f'  MAE  : ${mae:.2f}')
print(f'  RMSE : ${rmse:.2f}')
print(f'  MAPE : {mape:.2f}%')

<a name="skip-code-16"></a>

**Expected output:** Progress updates every 50 steps followed by final MAE, RMSE, and MAPE. For NVDA in the 2023–2024 test period, MAPE will likely be above 15% — the AI-driven surge creates a price level far above what the pre-2023 training data would predict.

<nav aria-label="Return navigation for code cell 16">
<a href="#read-code-16" aria-label="Go back and read code cell 16">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-17"></a>

**Cell 17 — Plot ARIMA Forecast vs Actual**

Two panels: the top shows the full price history with the walk-forward predictions overlaid on the test period. The bottom shows the forecast errors (actual minus predicted) as a bar chart. Results are stored in `arima_metrics` for the final comparison table.

<nav aria-label="Code cell 17 navigation">
<a href="#skip-code-17" aria-label="Skip code cell 17 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
pred_series = pd.Series(predictions, index=test.index)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(train.index, train, label="Training",      color="steelblue",  alpha=0.6)
axes[0].plot(test.index,  test,  label="Actual",        color="darkorange", linewidth=1.5)
axes[0].plot(test.index,  pred_series, label="ARIMA Forecast", color="green", linewidth=1.5, linestyle="--")
axes[0].set_title("ARIMA(1,1,1) Walk-Forward Forecast vs Actual — NVDA", fontsize=13)
axes[0].set_ylabel("Price (USD)")
axes[0].legend()

errors = actuals - predictions
axes[1].bar(test.index, errors, color="red", alpha=0.5, width=1)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Forecast Errors (Actual - Predicted)", fontsize=13)
axes[1].set_ylabel("Error (USD)")

plt.tight_layout()
plt.savefig('nvda_arima_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

arima_metrics = {"Model": "ARIMA(1,1,1)", "MAE": mae, "RMSE": rmse, "MAPE%": mape}

<a name="skip-code-17"></a>

**Expected output:** Two-panel chart. The forecast line will closely track the actual price (ARIMA mostly predicts 'tomorrow will be like today') but will systematically underpredict during NVDA's rapid ascent. The error bars will grow in magnitude during the 2023–2024 AI rally.

**Accessibility note:** Add a Markdown cell describing both panels — when are the errors largest, and what does that tell you about ARIMA's limitations?

<nav aria-label="Return navigation for code cell 17">
<a href="#read-code-17" aria-label="Go back and read code cell 17">&#8617; Go back and read the code cell</a>
</nav>

### Discussion Question 4A

> 1. Look at the ARIMA model summary from Cell 14. What do the AR and MA coefficients tell you? Are they statistically significant?

> 2. Walk-forward validation is more expensive than a single train/test split. When is the extra computational cost justified in a production trading system?

> 3. ARIMA essentially predicts 'tomorrow's price will be close to today's price plus a small mean-reversion correction.' Given this, what is the fundamental limitation of ARIMA for forecasting NVDA specifically — a stock whose price trajectory was radically reshaped by a macro event (the AI hardware boom)?

**Write your answers below:**

1.

2.

3.

---
# Section 5 — Time Series as Supervised Learning with sklearn

Here is a key paradigm shift: **you can reframe any time series problem as supervised regression** by engineering lag features.

If `price[t]` depends on `price[t-1], price[t-2], ..., price[t-n]`, those lagged values become features `X` and `price[t]` becomes the target `y`. Now you can use any sklearn model you already know. This 'sliding window' approach is widely used in production ML systems.

---

<a name="read-code-18"></a>

**Cell 18 — Engineer Lag Features**

This cell defines `make_lag_features()`, which converts the price series into a supervised learning dataset. For each row it creates 20 lagged price columns (lag_1 through lag_20) plus rolling mean and standard deviation features and calendar features (day of week, month). After the function runs, the output shape and feature list are printed.

<nav aria-label="Code cell 18 navigation">
<a href="#skip-code-18" aria-label="Skip code cell 18 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
def make_lag_features(series, lags=20, target_col='price'):
    data = pd.DataFrame(series.copy())
    data.columns = [target_col]
    for lag in range(1, lags + 1):
        data[f'lag_{lag}'] = data[target_col].shift(lag)
    data['rolling_mean_5']  = data[target_col].shift(1).rolling(5).mean()
    data['rolling_mean_20'] = data[target_col].shift(1).rolling(20).mean()
    data['rolling_std_5']   = data[target_col].shift(1).rolling(5).std()
    data['day_of_week']     = data.index.dayofweek
    data['month']           = data.index.month
    return data.dropna()

feat_df = make_lag_features(df['price'], lags=20)
print(f"Feature matrix shape: {feat_df.shape}")
print(f"Features: {list(feat_df.columns)}")
feat_df.head()

<a name="skip-code-18"></a>

**Expected output:** Shape of approximately (1,736, 26) — one row per trading day (minus the lag warmup period) and 26 columns (price target plus 20 lags plus 5 engineered features).

<nav aria-label="Return navigation for code cell 18">
<a href="#read-code-18" aria-label="Go back and read code cell 18">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-19"></a>

**Cell 19 — Chronological Train / Test Split for Sklearn**

This cell splits the lag-feature dataset into training and test sets using the same `SPLIT_DATE` as Section 4. The split uses `searchsorted` to find the exact index position — no random shuffling.

<nav aria-label="Code cell 19 navigation">
<a href="#skip-code-19" aria-label="Skip code cell 19 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
feature_cols = [c for c in feat_df.columns if c != 'price']
X = feat_df[feature_cols]
y = feat_df['price']

split_idx = feat_df.index.searchsorted(pd.Timestamp(SPLIT_DATE))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")

<a name="skip-code-19"></a>

**Expected output:** X_train with approximately 1,260 rows and X_test with approximately 476 rows.

<nav aria-label="Return navigation for code cell 19">
<a href="#read-code-19" aria-label="Go back and read code cell 19">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-20"></a>

**Cell 20 — Linear Regression Baseline on Lag Features**

Linear Regression with lag features is a fast, interpretable baseline. MAE, RMSE, MAPE, and R² are computed and printed.

<nav aria-label="Code cell 20 navigation">
<a href="#skip-code-20" aria-label="Skip code cell 20 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

lr_mae  = mean_absolute_error(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_mape = np.mean(np.abs((y_test.values - lr_preds) / y_test.values)) * 100
lr_r2   = r2_score(y_test, lr_preds)

print("Linear Regression on Lag Features")
print(f"  MAE  : ${lr_mae:.2f}")
print(f"  RMSE : ${lr_rmse:.2f}")
print(f"  MAPE : {lr_mape:.2f}%")
print(f"  R2   : {lr_r2:.4f}")

<a name="skip-code-20"></a>

**Expected output:** Metrics for the linear regression baseline. R² will likely be high (above 0.90) because `lag_1` is an extremely strong predictor of today's price — but this is a consequence of price persistence, not genuine forecasting skill.

<nav aria-label="Return navigation for code cell 20">
<a href="#read-code-20" aria-label="Go back and read code cell 20">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-21"></a>

**Cell 21 — Random Forest on Lag Features**

Random Forest captures non-linear relationships between lags and the current price. We use 200 trees with a max depth of 10 and all available CPU cores (`n_jobs=-1`).

<nav aria-label="Code cell 21 navigation">
<a href="#skip-code-21" aria-label="Skip code cell 21 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

rf_mae  = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_mape = np.mean(np.abs((y_test.values - rf_preds) / y_test.values)) * 100
rf_r2   = r2_score(y_test, rf_preds)

print("Random Forest on Lag Features")
print(f"  MAE  : ${rf_mae:.2f}")
print(f"  RMSE : ${rf_rmse:.2f}")
print(f"  MAPE : {rf_mape:.2f}%")
print(f"  R2   : {rf_r2:.4f}")

<a name="skip-code-21"></a>

**Expected output:** Random Forest metrics — typically lower MAPE than Linear Regression but potentially higher due to the out-of-distribution nature of NVDA's 2023–2024 AI surge. Tree-based models can struggle to extrapolate beyond the price range seen during training.

<nav aria-label="Return navigation for code cell 21">
<a href="#read-code-21" aria-label="Go back and read code cell 21">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-22"></a>

**Cell 22 — Random Forest Feature Importance**

This cell extracts and plots the top 15 most important features by mean decrease in impurity. This is one of the key advantages of tree-based methods over ARIMA — they tell you which lag distances matter most.

<nav aria-label="Code cell 22 navigation">
<a href="#skip-code-22" aria-label="Skip code cell 22 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
importances  = pd.Series(rf.feature_importances_, index=feature_cols)
top_features = importances.nlargest(15)

fig, ax = plt.subplots(figsize=(10, 6))
top_features.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Random Forest — Top 15 Feature Importances (NVDA Lag Model)', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('nvda_rf_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 5 most predictive features:")
print(top_features.head())

<a name="skip-code-22"></a>

**Expected output:** A horizontal bar chart. `lag_1` (yesterday's price) almost certainly dominates, followed by short rolling means. This reflects the strong autocorrelation in price levels — tomorrow's price is mostly determined by today's price, not complex patterns.

**Accessibility note:** Add a Markdown cell describing the chart. Note which feature is most important and what that implies about stock price dynamics.

<nav aria-label="Return navigation for code cell 22">
<a href="#read-code-22" aria-label="Go back and read code cell 22">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-23"></a>

**Cell 23 — Compare All Three Models Visually and in a Metrics Table**

This cell overlays Linear Regression, Random Forest, and ARIMA predictions on the same chart alongside the actual NVDA price. A formatted metrics table is also printed to enable direct comparison.

<nav aria-label="Code cell 23 navigation">
<a href="#skip-code-23" aria-label="Skip code cell 23 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(y_test.index, y_test.values,  label="Actual",       color="black",      linewidth=1.5)
ax.plot(y_test.index, lr_preds,        label="Lin. Reg.",    color="steelblue",  linewidth=1, linestyle="--")
ax.plot(y_test.index, rf_preds,        label="Rand. Forest", color="green",      linewidth=1, linestyle="-.")
ax.plot(test.index,   predictions,     label="ARIMA(1,1,1)", color="darkorange", linewidth=1, linestyle=":")
ax.set_title("Model Comparison — NVDA Test Period Forecasts", fontsize=14)
ax.set_ylabel("Price (USD)")
ax.legend()
plt.tight_layout()
plt.savefig('nvda_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

sklearn_metrics = [
    {"Model": "Linear Regression (lag)", "MAE": lr_mae,  "RMSE": lr_rmse,  "MAPE%": lr_mape},
    {"Model": "Random Forest (lag)",     "MAE": rf_mae,  "RMSE": rf_rmse,  "MAPE%": rf_mape},
]
metrics_table = pd.DataFrame([arima_metrics] + sklearn_metrics).set_index('Model')
print("\nModel Comparison Table:")
print(metrics_table.to_string())

<a name="skip-code-23"></a>

**Expected output:** An overlay chart showing how closely each model tracks the actual price, and a metrics table. All three models will likely underestimate NVDA's price during the AI-driven surge — this is expected and is a topic for Discussion Question 5A.

**Accessibility note:** Add a Markdown cell describing the chart. Which model tracks the actual price most closely? Where do all three models diverge most?

<nav aria-label="Return navigation for code cell 23">
<a href="#read-code-23" aria-label="Go back and read code cell 23">&#8617; Go back and read the code cell</a>
</nav>

### Discussion Question 5A

> 1. Random Forest may achieve a high R² score on NVDA lag features even though the model cannot 'know' about the AI boom. Why might R² be misleading here? What does a high R² on a price-level target actually tell you?

> 2. The feature importance plot almost certainly shows `lag_1` dominating. This means 'yesterday's price is the best predictor of today's price.' How does this relate to the Efficient Market Hypothesis (EMH)? If the market is semi-strong form efficient, what does that imply about the residual predictability these models are capturing?

> 3. If you were presenting these forecasts to a portfolio manager at a quantitative hedge fund, which model would you present and what additional evidence would they demand before trusting any model for trading decisions?

**Write your answers below:**

1.

2.

3.

---
# Section 6 — Forecasting with Prophet

**Prophet** was developed by Meta's Core Data Science team and open-sourced in 2017. It is designed for analysts who need reliable forecasts quickly without deep time series expertise. Prophet is widely used in industry for business metric forecasting, capacity planning, and anomaly detection baselines.

**Key strengths:**
- Handles missing data and outliers automatically
- Models trend, weekly seasonality, yearly seasonality, and holidays out of the box
- Interpretable components and built-in uncertainty intervals
- Robust to structural breaks through changepoint detection

Prophet expects a DataFrame with exactly two columns: `ds` (datestamp) and `y` (value).

---

<a name="read-code-24"></a>

**Cell 24 — Prepare Prophet DataFrame**

This cell reformats the price data into Prophet's required two-column structure (`ds` for dates, `y` for values), removes timezone information that can cause errors, and splits into train and test sets using the same split date.

<nav aria-label="Code cell 24 navigation">
<a href="#skip-code-24" aria-label="Skip code cell 24 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
prophet_df = df['price'].reset_index()
prophet_df.columns = ['ds', 'y']
prophet_df['ds'] = pd.to_datetime(prophet_df['ds'])

if prophet_df['ds'].dt.tz is not None:
    prophet_df['ds'] = prophet_df['ds'].dt.tz_localize(None)

prophet_train = prophet_df[prophet_df['ds'] < SPLIT_DATE]
prophet_test  = prophet_df[prophet_df['ds'] >= SPLIT_DATE]

print(f"Prophet train shape : {prophet_train.shape}")
print(f"Prophet test shape  : {prophet_test.shape}")
prophet_train.tail()

<a name="skip-code-24"></a>

**Expected output:** Prophet train shape of approximately (1,260, 2) and test shape of approximately (499, 2).

<nav aria-label="Return navigation for code cell 24">
<a href="#read-code-24" aria-label="Go back and read code cell 24">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-25"></a>

**Cell 25 — Fit Prophet Model**

This cell initializes Prophet with weekly and yearly seasonality enabled, a `changepoint_prior_scale` of 0.05 (controls how flexible the trend can be — higher values allow more breakpoints), US stock market holidays, and 95% uncertainty intervals. The `changepoint_prior_scale` is especially important for NVDA given its structural break in 2023.

<nav aria-label="Code cell 25 navigation">
<a href="#skip-code-25" aria-label="Skip code cell 25 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
prophet_model = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10,
    interval_width=0.95,
)

prophet_model.add_country_holidays(country_name='US')

print('Fitting Prophet model...')
prophet_model.fit(prophet_train)
print('Prophet model fitted.')

<a name="skip-code-25"></a>

**Expected output:** `Prophet model fitted.` Prophet will automatically detect changepoints in the training data — for NVDA it should identify the 2022 correction and the pre-2023 growth regime.

<nav aria-label="Return navigation for code cell 25">
<a href="#read-code-25" aria-label="Go back and read code cell 25">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-26"></a>

**Cell 26 — Generate Forecast**

This cell creates a future DataFrame covering the test period plus 90 additional business days, then runs Prophet's prediction. The output DataFrame contains `yhat` (point forecast), `yhat_lower` and `yhat_upper` (95% confidence interval bounds), `trend`, `weekly`, and `yearly` components.

<nav aria-label="Code cell 26 navigation">
<a href="#skip-code-26" aria-label="Skip code cell 26 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
future = prophet_model.make_future_dataframe(
    periods=len(prophet_test) + 90,
    freq='B'
)

forecast = prophet_model.predict(future)
forecast_test = forecast[forecast['ds'] >= SPLIT_DATE]

print(f"Forecast columns: {list(forecast.columns)}")
forecast[["ds", "yhat", "yhat_lower", "yhat_upper", "trend", "weekly", "yearly"]].tail(10)

<a name="skip-code-26"></a>

**Expected output:** A list of all Prophet output columns and a 10-row preview of the forecast table. Note that `yhat_lower` and `yhat_upper` define the uncertainty band — these should widen as the forecast extends further into the future.

<nav aria-label="Return navigation for code cell 26">
<a href="#read-code-26" aria-label="Go back and read code cell 26">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-27"></a>

**Cell 27 — Evaluate Prophet on the Test Set**

This cell aligns the Prophet predictions with the actual test values using a merge on the `ds` date column, then computes MAE, RMSE, and MAPE.

<nav aria-label="Code cell 27 navigation">
<a href="#skip-code-27" aria-label="Skip code cell 27 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
prophet_test_merged = prophet_test.merge(
    forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']], on='ds', how='inner'
)

p_mae  = mean_absolute_error(prophet_test_merged['y'], prophet_test_merged['yhat'])
p_rmse = np.sqrt(mean_squared_error(prophet_test_merged['y'], prophet_test_merged['yhat']))
p_mape = np.mean(np.abs((prophet_test_merged['y'] - prophet_test_merged['yhat']) / prophet_test_merged['y'])) * 100

print("Prophet Forecast Evaluation")
print(f"  MAE  : ${p_mae:.2f}")
print(f"  RMSE : ${p_rmse:.2f}")
print(f"  MAPE : {p_mape:.2f}%")

prophet_metrics = {"Model": "Prophet", "MAE": p_mae, "RMSE": p_rmse, "MAPE%": p_mape}

<a name="skip-code-27"></a>

**Expected output:** Evaluation metrics for Prophet. Given NVDA's post-2023 trajectory, all models including Prophet will show elevated MAPE — the AI-driven surge creates price levels well outside the training distribution. This is not a failure of the model; it reflects the fundamental unpredictability of macro events.

<nav aria-label="Return navigation for code cell 27">
<a href="#read-code-27" aria-label="Go back and read code cell 27">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-28"></a>

**Cell 28 — Prophet Component Plots**

Prophet's component plots are one of its most valuable features. They decompose the forecast into interpretable trend, weekly, and yearly seasonality components. For NVDA, the weekly component may show systematic patterns tied to options expiry cycles and institutional rebalancing, while the yearly component may reflect earnings seasonality.

<nav aria-label="Code cell 28 navigation">
<a href="#skip-code-28" aria-label="Skip code cell 28 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
fig = prophet_model.plot_components(forecast)
plt.suptitle('Prophet Forecast Components — NVDA', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

<a name="skip-code-28"></a>

**Expected output:** Component plots showing trend, weekly seasonality, and yearly seasonality. The trend should show a clear acceleration post-2023. The weekly pattern may show slightly lower expected returns on Mondays or Fridays — common in equity markets due to weekend information accumulation and institutional positioning.

**Accessibility note:** Add a Markdown cell describing each component plot. What business or market mechanics could explain the weekly and yearly patterns?

<nav aria-label="Return navigation for code cell 28">
<a href="#read-code-28" aria-label="Go back and read code cell 28">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-29"></a>

**Cell 29 — Full Forecast Plot with Uncertainty Intervals and 90-Day Ahead Forecast**

This cell creates the signature Prophet visualization: training history, actual test values, the within-sample forecast with 95% confidence band, and a 90-day out-of-sample forecast with its widening confidence band.

<nav aria-label="Code cell 29 navigation">
<a href="#skip-code-29" aria-label="Skip code cell 29 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(prophet_train["ds"], prophet_train["y"], color="steelblue", label="Training", alpha=0.7, linewidth=1)
ax.plot(prophet_test_merged["ds"], prophet_test_merged["y"], color="black", label="Actual (test)", linewidth=1.5)
ax.plot(forecast_test["ds"], forecast_test["yhat"], color="darkorange", label="Prophet Forecast", linewidth=1.5, linestyle="--")
ax.fill_between(forecast_test["ds"], forecast_test["yhat_lower"], forecast_test["yhat_upper"],
                alpha=0.2, color="darkorange", label="95% CI")

future_only = forecast[forecast["ds"] > prophet_df["ds"].max()]
ax.plot(future_only["ds"], future_only["yhat"], color="green", linewidth=1.5, linestyle="-.", label="90-day Ahead Forecast")
ax.fill_between(future_only["ds"], future_only["yhat_lower"], future_only["yhat_upper"], alpha=0.15, color="green")

ax.axvline(pd.Timestamp(SPLIT_DATE), color="red", linestyle="--", alpha=0.7, label="Split Date")
ax.set_title(f"Prophet Forecast — {TICKER}", fontsize=14)
ax.set_ylabel("Price (USD)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.savefig('nvda_prophet_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

<a name="skip-code-29"></a>

**Expected output:** A chart showing the training history, test period actuals (black), and Prophet's forecast (orange dashed) with uncertainty band, plus a green line extending 90 days beyond the last data point. The green band should visibly widen as it extends further — this honest representation of growing uncertainty is one of Prophet's most valuable features.

**Accessibility note:** Add a Markdown cell describing the chart, noting where the confidence bands are narrowest and widest and why.

<nav aria-label="Return navigation for code cell 29">
<a href="#read-code-29" aria-label="Go back and read code cell 29">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-30"></a>

**Cell 30 — Final Model Comparison Table (All Four Models)**

This cell assembles the metrics from all four models — ARIMA, Linear Regression, Random Forest, and Prophet — into a single formatted comparison table.

<nav aria-label="Code cell 30 navigation">
<a href="#skip-code-30" aria-label="Skip code cell 30 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
all_metrics = pd.DataFrame([
    arima_metrics,
    {"Model": "Linear Regression (lag)", "MAE": lr_mae,  "RMSE": lr_rmse,  "MAPE%": lr_mape},
    {"Model": "Random Forest (lag)",     "MAE": rf_mae,  "RMSE": rf_rmse,  "MAPE%": rf_mape},
    prophet_metrics,
]).set_index('Model').round(2)

print("\n" + "="*55 + "\n    COMPLETE MODEL COMPARISON TABLE\n" + "="*55)
print(all_metrics.to_string())
print('\nLower MAE / RMSE / MAPE% = better forecast accuracy')

<a name="skip-code-30"></a>

**Expected output:** A four-row comparison table. For NVDA's 2023–2024 test period, no model is likely to achieve MAPE below 10% — this is expected and is the honest result of forecasting a stock whose price trajectory was reshaped by an unprecedented macro event.

<nav aria-label="Return navigation for code cell 30">
<a href="#read-code-30" aria-label="Go back and read code cell 30">&#8617; Go back and read the code cell</a>
</nav>

### Discussion Question 6A

> 1. Prophet's uncertainty intervals widen as the forecast extends further into the future. Why? What are the practical implications for a business that needs to plan capacity based on 90-day NVDA price forecasts?

> 2. The weekly seasonality component in Prophet may show systematic patterns across days of the week. What market microstructure or institutional behavior could explain weekly patterns in NVDA's price? Think about options expiry (every Friday), earnings releases, and index rebalancing.

> 3. Looking at the final comparison table: if you had to choose ONE model for a production system that automatically generates daily NVDA price forecasts, which would you choose? Consider not just raw accuracy metrics but also interpretability, maintenance cost, and graceful degradation under regime change.

**Write your answers below:**

1.

2.

3.

---
# Section 7 — Agentic Forecasting Pipeline with Smolagents, GROQ, and HuggingFace

This is where everything comes together. We build an **AI agent** that autonomously:

1. Downloads and analyzes the NVDA time series
2. Runs a stationarity check
3. Fits an ARIMA model and generates a 30-day forecast
4. Narrates the results in plain English in the style of a professional analyst note

This mirrors a real **MLOps agentic pipeline** — the kind built at fintech firms, hedge funds, and data-driven enterprises.

### Architecture

```
User Query
    |
    v
CodeAgent (Smolagents)
    |
    |-- Tool: fetch_stock_data()
    |-- Tool: run_stationarity_check()
    |-- Tool: fit_arima_forecast()
    └-- LLM Narration  <-- GROQ / HuggingFace
```

---

<a name="read-code-31"></a>

**Cell 31 — Define the Three Agent Tools**

This cell defines three `@tool` decorated functions that the Smolagents CodeAgent can call. Each tool has type hints and a docstring — the agent reads these to understand what each tool does and when to invoke it. The tools encapsulate: data fetching, ADF stationarity testing, and ARIMA fitting with forecast generation.

<nav aria-label="Code cell 31 navigation">
<a href="#skip-code-31" aria-label="Skip code cell 31 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
@tool
def fetch_stock_data(ticker: str, start: str, end: str) -> str:
    """
    Fetches daily closing price data for a stock ticker from Yahoo Finance.
    Returns a summary string with basic statistics.

    Args:
        ticker: Stock ticker symbol (e.g., "NVDA", "AAPL")
        start:  Start date in YYYY-MM-DD format
        end:    End date in YYYY-MM-DD format
    """
    import yfinance as yf
    import numpy as np
    data = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if data.empty:
        return f'ERROR: No data found for {ticker}.'
    close = data["Close"].squeeze()
    returns = close.pct_change().dropna()
    return (
        f"Ticker: {ticker}\n"
        f"Date Range: {close.index.min().date()} to {close.index.max().date()}\n"
        f"Observations: {len(close)}\n"
        f"Price Range: ${close.min():.2f} - ${close.max():.2f}\n"
        f"Mean Price: ${close.mean():.2f}\n"
        f"Std Dev: ${close.std():.2f}\n"
        f"Mean Daily Return: {returns.mean()*100:.4f}%\n"
        f"Annualized Volatility: {returns.std()*np.sqrt(252)*100:.2f}%\n"
        f"Total Return over period: {((close.iloc[-1]/close.iloc[0])-1)*100:.2f}%"
    )


@tool
def run_stationarity_check(ticker: str, start: str, end: str) -> str:
    """
    Downloads stock data and runs an Augmented Dickey-Fuller stationarity test.
    Returns the test result and recommended differencing order d.

    Args:
        ticker: Stock ticker symbol
        start:  Start date YYYY-MM-DD
        end:    End date YYYY-MM-DD
    """
    import yfinance as yf
    from statsmodels.tsa.stattools import adfuller
    data  = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    close = data["Close"].squeeze().dropna()
    result_raw  = adfuller(close)
    result_diff = adfuller(close.diff().dropna())
    d = 0 if result_raw[1] < 0.05 else 1
    return (
        f"ADF Test - {ticker} Raw Price:\n"
        f"  p-value: {result_raw[1]:.6f} -> {'STATIONARY' if result_raw[1]<0.05 else 'NON-STATIONARY'}\n\n"
        f"ADF Test - {ticker} First Difference:\n"
        f"  p-value: {result_diff[1]:.6f} -> {'STATIONARY' if result_diff[1]<0.05 else 'NON-STATIONARY'}\n\n"
        f"Recommended d (differencing order): {d}"
    )


@tool
def fit_arima_and_forecast(ticker: str, start: str, end: str,
                            p: int, d: int, q: int, forecast_days: int) -> str:
    """
    Fits an ARIMA(p,d,q) model and generates a forecast.
    Returns model performance and future price predictions.

    Args:
        ticker:        Stock ticker symbol
        start:         Training start date (YYYY-MM-DD)
        end:           Training end date (YYYY-MM-DD)
        p:             AR order
        d:             Differencing order
        q:             MA order
        forecast_days: Number of trading days to forecast
    """
    import yfinance as yf
    import numpy as np
    from statsmodels.tsa.arima.model import ARIMA
    data  = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    close = data["Close"].squeeze().dropna()
    model  = ARIMA(close, order=(p, d, q))
    result = model.fit()
    forecast    = result.forecast(steps=forecast_days)
    conf_int    = result.get_forecast(steps=forecast_days).conf_int(alpha=0.05)
    last_price  = close.iloc[-1]
    forecast_lines = []
    for i, (yhat, (lo, hi)) in enumerate(zip(forecast, conf_int.values), 1):
        forecast_lines.append(f'  Day +{i:02d}: ${yhat:.2f}  (95% CI: ${lo:.2f} - ${hi:.2f})')
    return (
        f"ARIMA({p},{d},{q}) Model - {ticker}\n" + "="*50 + "\n"
        f"AIC: {result.aic:.2f}  |  BIC: {result.bic:.2f}\n"
        f"Last Observed Price: ${last_price:.2f}\n\n"
        f"{forecast_days}-Day Ahead Forecast:\n" + "\n".join(forecast_lines[:10])
        + f"\n  ...\n  Day +{forecast_days}: ${forecast.iloc[-1]:.2f}"
        + f"\n\nImplied {forecast_days}-day Return: {((forecast.iloc[-1]/last_price)-1)*100:.2f}%"
    )

print("All agent tools defined.")
print("   Tools: fetch_stock_data, run_stationarity_check, fit_arima_and_forecast")

<a name="skip-code-31"></a>

**Expected output:** `All agent tools defined.`

Each tool is a self-contained Python function the agent can call by name. The docstrings and type hints are not decorative — the Smolagents framework reads them to generate the agent's tool-use reasoning.

<nav aria-label="Return navigation for code cell 31">
<a href="#read-code-31" aria-label="Go back and read code cell 31">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-32"></a>

**Cell 32 — Initialize the GROQ LLM Backend**

GROQ provides blazing-fast inference of open-source LLMs on custom ASIC hardware. We use `OpenAIServerModel` because GROQ exposes an OpenAI-compatible REST API. The model is Llama 3.3 70B — a strong open-source model available on GROQ's free tier.

<nav aria-label="Code cell 32 navigation">
<a href="#skip-code-32" aria-label="Skip code cell 32 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from smolagents import OpenAIServerModel, CodeAgent

groq_model = OpenAIServerModel(
    model_id = "llama-3.3-70b-versatile",
    api_base = "https://api.groq.com/openai/v1",
    api_key  = GROQ_API_KEY,
)

print("GROQ backend initialized.")
print("   Model: llama-3.3-70b-versatile via GROQ")

<a name="skip-code-32"></a>

**Expected output:** `GROQ backend initialized.`

If you see an authentication error, verify that your GROQ_API_KEY is correctly stored in Colab Secrets.

<nav aria-label="Return navigation for code cell 32">
<a href="#read-code-32" aria-label="Go back and read code cell 32">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-33"></a>

**Cell 33 — Initialize the HuggingFace LLM Backend**

HuggingFace Inference API provides free access to thousands of open-source models. We use Qwen2.5-72B-Instruct — a strong open-source model with excellent instruction following. The `InferenceClientModel` wrapper makes it compatible with the same Smolagents interface as GROQ.

<nav aria-label="Code cell 33 navigation">
<a href="#skip-code-33" aria-label="Skip code cell 33 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from smolagents import InferenceClientModel

hf_model = InferenceClientModel(
    model_id = "Qwen/Qwen2.5-72B-Instruct",
    token    = HF_TOKEN,
)

print("HuggingFace backend initialized.")
print("   Model: Qwen/Qwen2.5-72B-Instruct via HuggingFace Inference API")

<a name="skip-code-33"></a>

**Expected output:** `HuggingFace backend initialized.`

If you see a rate limit error, wait 60 seconds and retry — the free tier has request limits.

<nav aria-label="Return navigation for code cell 33">
<a href="#read-code-33" aria-label="Go back and read code cell 33">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-34"></a>

**Cell 34 — Assemble and Run the Forecasting Agent (GROQ)**

This cell creates a `CodeAgent` bound to the three tools and the GROQ model backend, then sends it a natural language task. The agent will reason about which tools to call, in what order, and how to synthesize the results into an analyst report. With `verbosity_level=2` you will see the agent's full reasoning trace — a valuable window into how agentic reasoning works.

<nav aria-label="Code cell 34 navigation">
<a href="#skip-code-34" aria-label="Skip code cell 34 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
agent_tools = [fetch_stock_data, run_stationarity_check, fit_arima_and_forecast]

forecasting_agent = CodeAgent(
    tools=agent_tools,
    model=groq_model,
    max_steps=8,
    verbosity_level=2,
)

AGENT_QUERY = f"""
You are a senior quantitative analyst covering semiconductor and AI infrastructure stocks.
Analyze NVDA (NVIDIA Corporation) using the following steps:

1. Fetch and summarize the stock data from 2018-01-01 to 2024-12-31.
2. Run a stationarity test and determine the appropriate differencing order (d).
3. Fit an ARIMA(1, d, 1) model and generate a 30-day ahead price forecast.
4. Write a concise, professional analyst report (3-5 paragraphs) covering:
   - Key statistics and trend observations (note the AI-driven acceleration from 2023)
   - Stationarity findings and what they mean for ARIMA modeling
   - The 30-day forecast and implied return
   - One key risk caveat for investors (e.g., concentration risk, valuation multiples)
   - A clear bottom-line view on the stock's near-term momentum

Write in the style of a Bloomberg Intelligence analyst note.
"""

print("Agent starting NVDA analysis...\n" + "="*60)
agent_report = forecasting_agent.run(AGENT_QUERY)
print("\n" + "="*60 + "\nFINAL AGENT REPORT\n" + "="*60)
print(agent_report)

<a name="skip-code-34"></a>

**Expected output:** The agent's full reasoning trace (tool calls, code execution, results) followed by a professionally written 3–5 paragraph analyst note on NVDA. The report should reference the statistical findings from the tools and synthesize them into a narrative suitable for an institutional audience.

<nav aria-label="Return navigation for code cell 34">
<a href="#read-code-34" aria-label="Go back and read code cell 34">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-35"></a>

**Cell 35 — Re-run with HuggingFace Backend**

One of the most important architectural principles in production ML is **provider abstraction** — your pipeline should not be locked to a single vendor. This cell re-runs the same task using Qwen2.5-72B via HuggingFace instead of Llama via GROQ. Compare the outputs for reasoning quality, writing style, and consistency.

<nav aria-label="Code cell 35 navigation">
<a href="#skip-code-35" aria-label="Skip code cell 35 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
forecasting_agent_hf = CodeAgent(
    tools=agent_tools,
    model=hf_model,
    max_steps=8,
    verbosity_level=0,
)

print("HuggingFace agent starting NVDA analysis...\n")
hf_report = forecasting_agent_hf.run(AGENT_QUERY)
print("\n" + "="*60 + "\nHUGGINGFACE AGENT REPORT\n" + "="*60)
print(hf_report)

<a name="skip-code-35"></a>

**Expected output:** A second analyst note generated by a different model. Compare with the GROQ output — do both models reach the same conclusion about the 30-day forecast? Do they differ in writing style, risk framing, or level of statistical detail?

<nav aria-label="Return navigation for code cell 35">
<a href="#read-code-35" aria-label="Go back and read code cell 35">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-36"></a>

**Cell 36 — Direct GROQ API Call for Model Comparison Narration**

Beyond agentic use, you can call GROQ directly using the standard chat completions API. This is useful when you already have structured results and just need an LLM to narrate them. This cell passes the full model comparison table to GROQ and asks for a concise 3-sentence assessment.

<nav aria-label="Code cell 36 navigation">
<a href="#skip-code-36" aria-label="Skip code cell 36 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from groq import Groq as GroqClient

groq_client = GroqClient(api_key=GROQ_API_KEY)
metrics_text = all_metrics.to_string()

groq_chat = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a senior data scientist writing for a financial audience. Be concise and precise."},
        {"role": "user", "content": (
            f"Here are the results from four forecasting models applied to {TICKER} stock:\\n\\n{metrics_text}\\n\\n"
            "In 3 sentences, explain which model performed best, why that might be, "
            "and what a practitioner should consider before deploying any of these models in a live trading system."
        )}
    ],
    temperature=0.3,
    max_tokens=400,
)

print("GROQ Direct Response:")
print("-" * 50)
print(groq_chat.choices[0].message.content)

<a name="skip-code-36"></a>

**Expected output:** A 3-sentence assessment comparing the four models. Notice that a direct LLM call is simpler and faster than using an agent — agents are only worth the overhead when you need multi-step reasoning and tool use.

<nav aria-label="Return navigation for code cell 36">
<a href="#read-code-36" aria-label="Go back and read code cell 36">&#8617; Go back and read the code cell</a>
</nav>

<a name="read-code-37"></a>

**Cell 37 — Direct HuggingFace Inference API Call**

This cell makes the equivalent direct call using the HuggingFace InferenceClient. The same prompt is sent to Qwen2.5-72B — compare the response with GROQ's for differences in reasoning style, specificity, and accuracy.

<nav aria-label="Code cell 37 navigation">
<a href="#skip-code-37" aria-label="Skip code cell 37 and go to explanation">&#9197; Skip this code cell</a>
</nav>

In [ ]:
from huggingface_hub import InferenceClient

hf_client = InferenceClient(token=HF_TOKEN)

hf_response = hf_client.chat_completion(
    model="Qwen/Qwen2.5-72B-Instruct",
    messages=[
        {"role": "system", "content": "You are a concise financial data science assistant."},
        {"role": "user", "content": (
            f"Model comparison for {TICKER} forecasting:\\n{metrics_text}\\n\\n"
            "In 3 sentences: which model wins, the most likely reason, and one critical production caveat."
        )}
    ],
    max_tokens=400,
    temperature=0.3,
)

print("HuggingFace Direct Response:")
print("-" * 50)
print(hf_response.choices[0].message.content)

<a name="skip-code-37"></a>

**Expected output:** A 3-sentence assessment from Qwen2.5-72B. Save both the GROQ and HuggingFace responses — you will reference them in Discussion Question 7A.

<nav aria-label="Return navigation for code cell 37">
<a href="#read-code-37" aria-label="Go back and read code cell 37">&#8617; Go back and read the code cell</a>
</nav>

### Discussion Question 7A — The Big Picture

> These are the capstone questions for this workshop. They represent the level of depth expected in a senior data science interview.

> 1. **Agentic Design:** Our agent used three tools. If you were extending this pipeline for a production trading desk at a firm like Two Sigma or Citadel, what additional tools would you add? Think about real-time data feeds, risk limit checks, compliance flags, and model versioning.

> 2. **LLM Provider Trade-offs:** You just ran the same task on GROQ (Llama 3.3 70B) and HuggingFace (Qwen2.5 72B). What differences did you observe in output quality, reasoning depth, and writing style? In a regulated financial environment, what criteria beyond performance would determine which provider you choose?

> 3. **The Limits of Forecasting:** All four models were applied to NVDA during the 2023–2024 AI hardware boom — one of the most extreme regime changes in recent market history. Given the structural break you observed in the data, what is your honest assessment of the practical utility of any of these models? Under what narrower conditions might time series forecasting genuinely add value for NVDA specifically?

> 4. **Connecting to the Previous Module:** How does the agentic pipeline we built today relate to the Federated Learning paradigm from the ML Paradigms notebook? Could you design a federated forecasting agent where each trading desk contributes model updates without sharing proprietary position data? What would be the technical and regulatory challenges?

> 5. **Capstone Connection:** How could you apply the techniques in this notebook to your own final project? Identify one specific component — a tool, a model, or an agentic pattern — that you could adapt for your research question.

**Write your answers below (double-click to edit):**

1.

2.

3.

4.

5.

---
# Workshop Summary

Congratulations — you have built a complete, industry-grade time series forecasting and agentic pipeline.

| Section | What You Built | Industry Skill |
| :--- | :--- | :--- |
| 1 | Reproducible environment setup | Dependency management |
| 2 | Live NVDA price acquisition and EDA | `yfinance`, exploratory analysis |
| 3 | Stationarity testing and decomposition | ADF test, ACF/PACF interpretation |
| 4 | ARIMA with walk-forward validation | Classical time series modeling |
| 5 | Lag-feature regression (LR and RF) | Supervised ML on time series |
| 6 | Prophet with uncertainty intervals | Production-grade forecasting |
| 7 | Agentic pipeline (GROQ and HuggingFace) | LLM orchestration, tool use, MLOps |

## Key Interview Concepts to Remember

- **Stationarity is a prerequisite**, not a suggestion. Always ADF-test before modeling.
- **Never randomly split time series** — always use chronological splits.
- **Walk-forward validation** is the gold standard for time series evaluation.
- **Lag features** bridge time series and supervised learning — any sklearn model can be applied.
- **Agentic pipelines** are the future of production ML — tools plus LLMs plus orchestration.
- **Provider abstraction** — your code should not be locked to one LLM vendor.
- **Regime changes break models** — always test on out-of-distribution data and interpret MAPE honestly.

## Further Reading

- Box, Jenkins et al. — *Time Series Analysis: Forecasting and Control* (the ARIMA reference)
- Taylor and Letham — *Forecasting at Scale* (the original Prophet paper, 2018)
- HuggingFace Smolagents docs — [huggingface.co/docs/smolagents](https://huggingface.co/docs/smolagents)
- GROQ docs — [console.groq.com/docs](https://console.groq.com/docs)

---

**To share:** Go to **Share** in Colab → set access to **Anyone with the link can view** → copy the link → paste into Canvas.